# 06-FE2 — Pair target encoding ablation

Notebook 04と同じfold・モデル設定を使い、base特徴量へNested Pair Target Encoding 120列を
追加したときの差だけを測ります。比較対象はXGBoost、Logistic Regression、MLPです。

In [ ]:
from functools import partial
from pathlib import Path
import sys
import time

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import get_base_features, load_nested_pair_te
from train import make_model, run_cv

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN

train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0})
folds = pd.read_csv(ROOT / "output" / "stkfolds.csv")
train = train.merge(folds, on=ID_COLUMN, how="left", validate="one_to_one")
assert train[[TARGET, FOLD_COLUMN]].notna().all().all()

## feature schema

TE列は数値として扱い、カテゴリ列のone-hot処理とは分離します。

In [ ]:
BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
FEATURE_DIR = ROOT / "output" / "nested_pair_te"
sample_te = pd.read_parquet(FEATURE_DIR / "outer_fold_0_train.parquet")
PAIR_TE_FEATURES = [column for column in sample_te.columns if column != ID_COLUMN]
NUM_FEATURES = BASE_NUM + PAIR_TE_FEATURES
CAT_FEATURES = BASE_CAT

assert len(PAIR_TE_FEATURES) == 120
assert not (set(BASE_FEATURES) & set(PAIR_TE_FEATURES))
print("base numeric:", len(BASE_NUM))
print("base categorical:", len(BASE_CAT))
print("pair TE:", len(PAIR_TE_FEATURES))
print("model inputs:", len(NUM_FEATURES) + len(CAT_FEATURES))

In [ ]:
pair_te_loader = partial(load_nested_pair_te, feature_dir=FEATURE_DIR)
MODEL_ORDER = ["XGBoost", "LogisticRegression", "MLP"]
rows = []
results = {}

for model_name in MODEL_ORDER:
    print("=" * 60)
    print("Running:", model_name)
    started = time.time()
    model = make_model(
        model_name,
        NUM_FEATURES,
        CAT_FEATURES,
        seed=Baseline.SEED,
        use_gpu=False,
    )
    prefix = f"fe2_{model_name.lower().replace(' ', '_')}"
    result = run_cv(
        model=model,
        train=train,
        test=test,
        features=BASE_FEATURES,
        target=TARGET,
        id_column=ID_COLUMN,
        fold_column=FOLD_COLUMN,
        label=f"{model_name}+nested_pair_te",
        save_prefix=prefix,
        output_dir=ROOT / "artifacts",
        fold_feature_loader=pair_te_loader,
    )
    elapsed = time.time() - started
    rows.append(
        {
            "model": model_name,
            "fe2_oof_auc": result["oof_auc"],
            "fold_auc_mean": result["fold_df"]["auc"].mean(),
            "fold_auc_std": result["fold_df"]["auc"].std(ddof=0),
            "time_sec": elapsed,
        }
    )
    results[model_name] = result

fe2_comparison = pd.DataFrame(rows).sort_values("fe2_oof_auc", ascending=False)
display(fe2_comparison.reset_index(drop=True))

## baseとの差を自動表示

絶対スコアだけでなく、同じモデル・同じfoldに対する差分で特徴量の効果を判断します。

In [ ]:
comparison_rows = []
for model_name in MODEL_ORDER:
    baseline_prefix = model_name.lower().replace(" ", "_")
    baseline_summary = pd.read_csv(
        ROOT / "artifacts" / f"{baseline_prefix}_summary.csv"
    ).iloc[0]
    fe2_auc = results[model_name]["oof_auc"]
    comparison_rows.append(
        {
            "model": model_name,
            "baseline_oof_auc": baseline_summary["oof_auc"],
            "fe2_oof_auc": fe2_auc,
            "delta": fe2_auc - baseline_summary["oof_auc"],
        }
    )

ablation = pd.DataFrame(comparison_rows).sort_values("delta", ascending=False)
display(ablation)

## 判断方法

- 全foldで同じ向きに動くかを確認します。
- 改善がモデルごとに違う場合、TE特徴量を全モデルへ一律採用しません。
- 線形モデルだけ改善するなら、カテゴリ交互作用を明示した効果と解釈できます。
- XGBoostやMLPが悪化する場合は、120列を全投入せずpair選択、正則化、モデル別採否を検討します。
- `fe2_*` という別prefixで予測を保存するため、base artifactを上書きしません。